In [0]:
dbutils.fs.ls("/Volumes/workspace/default/movielens_volume/")


In [0]:
ratings_df = spark.read.csv(
    "/Volumes/workspace/default/movielens_volume/ratings.dat",
    sep="::",
    inferSchema=True
).toDF("userId", "movieId", "rating", "timestamp")

movies_df = spark.read.csv(
    "/Volumes/workspace/default/movielens_volume/movies.dat",
    sep="::",
    inferSchema=True
).toDF("movieId", "title", "genres")


In [0]:
ratings_df.printSchema()
movies_df.printSchema()


In [0]:
dbutils.fs.ls("/Volumes/workspace/default/movielens_volume/")

In [0]:
movies_df = spark.read \
    .option("delimiter", "::") \
    .option("inferSchema", "true") \
    .csv("dbfs:/Volumes/workspace/default/movielens_volume/movies.dat")


In [0]:
movies_df.show(5)

In [0]:
movies_df.printSchema()

In [0]:
movies_df = movies_df.toDF("movie_id", "title", "genres")


In [0]:
movies_df.show(5)

In [0]:
movies_df.createOrReplaceTempView("movies")

display(movies_df)

In [0]:
spark.sql("select * from movies").show(5)


In [0]:
result_df = spark.sql(""" SELECT movie_id,title from movies LIMIT 10""")
result_df.show()

In [0]:
ratings_df = spark.read \
  .option("delimiter", "::") \
  .option("inferSchema", "true") \
  .csv("dbfs:/Volumes/workspace/default/movielens_volume/ratings.dat") \
  .toDF("user_id", "movie_id", "rating", "timestamp")


In [0]:
ratings_df.createOrReplaceTempView("ratings")
display(ratings_df)

In [0]:
spark.sql("""
SELECT
  user_id,
  movie_id,
  rating,
  FROM_UNIXTIME(timestamp) AS rating_time
FROM ratings
LIMIT 10
""").show()

In [0]:
spark.sql("select * from ratings").show()

In [0]:
spark.sql("""SELECT timestamp,FROM_UNIXTIME(timestamp) AS rating_time FROM ratings LIMIT 5""").show()

In [0]:
spark.sql("""CREATE OR REPLACE TEMP VIEW clean_ratings AS SELECT user_id,movie_id,rating,FROM_UNIXTIME(timestamp) AS rating_time FROM ratings  where rating IS NOT NULL""")

In [0]:
spark.sql("""
SELECT *
FROM clean_ratings
LIMIT 5
""").show()


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW clean_ratings AS
SELECT
  user_id,
  movie_id,
  rating,
  CAST(FROM_UNIXTIME(timestamp) AS TIMESTAMP) AS rating_ts,
  DATE(FROM_UNIXTIME(timestamp))AS rating_date,
  date_format(rating_ts, 'HH:mm:ss') AS rating_time
FROM ratings
WHERE rating IS NOT NULL
""").show()


In [0]:
spark.sql("DESCRIBE clean_ratings").show(truncate=False)


In [0]:
tags_df = spark.read \
  .option("delimiter", "::") \
  .option("inferSchema", "true") \
  .csv("dbfs:/Volumes/workspace/default/movielens_volume/tags.dat") \
  .toDF("user_id", "movie_id", "tag", "timestamp")

tags_df.createOrReplaceTempView("tags")


In [0]:
from pyspark.sql.functions import from_unixtime, to_date, date_format
tags_enriched_df = tags_df \
    .withColumn("tags_ts", from_unixtime("timestamp")) \
    .withColumn("tags_date", to_date("tags_ts")) \
    .withColumn("tags_time", date_format("tags_ts", "HH:mm:ss"))

tags_enriched_df.createOrReplaceTempView("tags")



In [0]:
spark.sql("SHOW TABLES").show()

In [0]:
spark.sql("""select * from tags LIMIT 5""").show()

In [0]:
spark.sql("DESCRIBE movies").show(truncate=False)
spark.sql("DESCRIBE ratings").show(truncate=False)
spark.sql("DESCRIBE clean_ratings").show(truncate=False)
spark.sql("DESCRIBE tags").show(truncate=False)

In [0]:
spark.sql("""CREATE OR REPLACE TEMP VIEW tags_with_time AS SELECT user_id,movie_id,tag,timestamp,CAST(FROM_UNIXTIME(timestamp) AS TIMESTAMP) AS tag_time FROM tags""")

In [0]:
spark.sql("DESCRIBE tags_with_time").show(truncate=False)

# PART 1 – Basic Exploration (Spark SQL Case Study)


In [0]:
-- 1. How many total users are in the Users table?
spark.sql("""select count(distinct user_id) AS total_user from clean_ratings""").show()


In [0]:
# 2.Count the total number of movies available in the dataset.
spark.sql("""select count(distinct movie_id)AS total_movies from movies""").show()

In [0]:
# Q1. What is the total number of ratings in the dataset?
# Determine how many rating records are available.
# Hint: Use COUNT(*) on the ratings view.
spark.sql("""select count(*) as Total_rating from clean_ratings""").show()

In [0]:
# Q2. How many distinct users have provided ratings?
# Count each user only once, regardless of how many ratings they gave.
# Hint: Use COUNT(DISTINCT userId).
spark.sql("""select count(distinct user_id) as total from clean_ratings""").show()

In [0]:
# Q3. How many distinct movies have been rated?
# Count only those movies that appear in the ratings data.
# Hint: Use COUNT(DISTINCT movieId).
spark.sql("""select count(distinct movie_id) as total from clean_ratings""").show()

In [0]:
# Q4. What is the overall rating scale?
# Find the minimum rating, maximum rating, and average rating across the dataset.
# Hint: Use MIN(rating), MAX(rating), and AVG(rating).
spark.sql("""select MIN(rating)as Min_rating,MAX(rating)as MAX_rating,AVG(rating)as AVG_rating from clean_ratings""").show()

In [0]:
# Q5. Which movies are the most popular based on number of ratings?
# List the top 10 movies with the highest rating counts.
# Hint: Group by movieId and order by COUNT(*) in descending order.
spark.sql("""select  count(*), movie_id from clean_ratings group by movie_id order by count(*) DESC LIMIT 10""").show()

In [0]:
# Q6. On average, how many ratings does a user give?
# Measure overall user engagement on the platform.
# Hint: Total ratings divided by distinct users, or use a subquery.
spark.sql("""select count(*)/count(distinct user_id) as user_engagement from clean_Ratings""").show()

In [0]:
spark.sql("DESCRIBE clean_ratings").show()

In [0]:
# Q7. How many ratings are submitted each day?
# Analyze rating activity at a daily level.
# Hint: Group by rating_date.
spark.sql("SELECT rating_date ,COUNT(*) AS total_ratings FROM clean_ratings GROUP BY rating_date ORDER BY rating_date").show()


In [0]:
# Q8. Which days have the highest rating activity?
# Identify the top 5 days with the maximum number of ratings.
# Hint: Order daily counts in descending order.
spark.sql("""select count(*),rating_date as rating_date from clean_Ratings group by rating_date order by count(*) DESC LIMIT 5""").show()

In [0]:
# Q9. What does the rating distribution look like?
# Count how many ratings exist for each rating value (1.0, 1.5, 2.0 … 5.0).
# Hint: Group by rating.
spark.sql("select count(*) as total,rating from clean_Ratings group by rating").show()

In [0]:
# Q10. Are there movies that were rated only once?
# Identify movies with very low engagement.
# Hint: Use GROUP BY movieId HAVING COUNT(*) = 1.
spark.sql("""select count(*) as total,movie_id from clean_Ratings group by movie_id having count(*) = 1 """).show()

In [0]:
spark.sql("show tables").show()

In [0]:
spark.sql("select * from clean_ratings")

## PART 2 – Data Cleaning & Feature Engineering

In [0]:
# Q1. Are there any NULL or missing values in critical columns?
# Check for NULLs in user_id, movie_id, rating, title, and genres.
# Hint:
# Use COUNT(*) with IS NULL conditions per column.
spark.sql("""select SUM(CASE WHEN c.user_id IS NULL THEN 1 ELSE 0 END) AS user_id_nulls ,
                    SUM(CASE WHEN m.movie_id IS NULL THEN 1 ELSE 0 END) AS MOVIE_ID_NULLS,
                    SUM(CASE WHEN c.rating IS NULL THEN 1 ELSE 0 END )AS Rating ,
                    SUM(CASE WHEN m.title IS NULL THEN 1 ELSE 0 END) AS title_nulls,
                    SUM(CASE WHEN m.genres IS NULL THEN 1 ELSE 0 END) AS genres_nulls
                  from clean_ratings  c join movies  m on c.movie_id = m.movie_id""").show()  

In [0]:
spark.sql("""select *
                  from clean_ratings  c join movies  m on c.movie_id = m.movie_id where m.genres is null""").show()  

In [0]:
# Q2. Are there duplicate ratings for the same user and movie at the same timestamp?
# Such duplicates can distort analytics.
# Hint:
# GROUP BY user_id, movie_id, timestamp HAVING COUNT(*) > 1
spark.sql("""select count(*)as total,rating ,user_id,movie_id,rating_time from clean_ratings group by user_id,movie_id,rating_time,rating having count(*)>1""").show()

In [0]:
# Q3. Validate rating values.
# Ensure all ratings fall within the valid range (0.5 to 5.0).
# Hint:
# Filter ratings < 0.5 OR rating > 5.0
spark.sql("""select count(*) as total ,rating from clean_ratings where rating <0.5 or rating >5.0 group by rating""").show()

In [0]:
# 4. Validate timestamp consistency.
# Check if rating_ts matches the Unix timestamp column logically.
# Hint:
# Compare from_unixtime(timestamp) with rating_ts
display(
spark.sql("""
SELECT count(*) AS mismatched_timestamps
FROM clean_ratings
WHERE date_format(rating_ts, 'HH:mm:ss') != rating_time
""")
)

In [0]:
spark.sql("select * from tags")

In [0]:
spark.sql("select * from tags").show()

In [0]:
spark.sql("""
SELECT count(*) AS mismatched_timestamps
FROM tags
WHERE date_format(tags_ts, 'HH:mm:ss') != tags_time
""").show()

In [0]:
# Q5. Standardize timestamp columns.
# Convert rating_ts and tags_ts from STRING to TIMESTAMP if required.
# Hint:
# CAST or to_timestamp functions


In [0]:
# Q6. Clean and normalize tags. 
# Identify tags with:
# - Leading/trailing spaces
# - Upper/lower case inconsistencies
# - Empty or meaningless values
# Hint:
# Use trim(tag), lower(tag), and filter empty strings
spark.sql("""select trim(tag) as tag_trim,lower(tag)as lower_tag,upper(tag) from tags where tag is null""").show()

In [0]:
# Q7. Detect movies with missing or invalid genres.
# Some movies may have NULL or empty genre values.
# Hint:
# Check genres IS NULL or genres = ''
spark.sql("""select movie_id ,genres from movies where genres is NULL or genres = '' """).show()

In [0]:
# Q8. Split multi-valued genres into individual rows.
# Prepare the data for genre-level analytics.
# Hint:
# Use split(genres, '\\|') and explode()
spark.sql(""" select movie_id,title,EXPLODE(SPLIT(genres, '\\\\|')) AS genre from movies """).show(truncate=False)

In [0]:
# Q9. Extract release year from movie titles.
# The year is embedded inside parentheses in the title.
# Hint:
# Use regexp_extract(title, '\\((\\d{4})\\)', 1)
spark.sql("""SELECT movie_id, title,regexp_extract(title, '\\\\((\\\\d{4})\\\\)', 1) AS release_year FROM movies""").show(truncate=False)


In [0]:
spark.sql("show tables").show()

In [0]:
spark.sql("select * from movies").show()

In [0]:
# Q10. Create clean analytical views.
# Build cleaned views for downstream analysis:
# - ratings_clean
# - tags_clean
# - movies_clean
# - movie_genres (flattened)
# Hint:
# Use CREATE OR REPLACE TEMP VIEW with cleaned logic
spark.sql(""" CREATE OR REPLACE TEMP VIEW cleaned_logic as select movie_id,title,genres from movies """).show()

In [0]:
spark.sql("""CREATE OR REPLACE TEMP VIEW ratings_clean as select * from ratings""")
spark.sql("""
CREATE OR REPLACE TEMP VIEW movie_genres AS
SELECT
    movie_id,
    title,
    EXPLODE(SPLIT(genres, '\\\\|')) AS genre
FROM movies
""")


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW movies_clean AS
SELECT
    movie_id,
    title,
    genres,
    CAST(regexp_extract(title, '\\\\((\\\\d{4})\\\\)', 1) AS INT) AS release_year
FROM movies
""")


In [0]:
spark.sql("""CREATE OR REPLACE TEMP VIEW tags_clean as select * from tags""").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW ratings_clean AS
SELECT * FROM clean_ratings""")


In [0]:
spark.sql("show tables").show()

In [0]:
display(spark.sql("SHOW TABLES"))

In [0]:
spark.sql("DROP VIEW IF EXISTS cleaned_logic")

### PART 3 – Post-Cleaning Join Analysis & Core Analytics

In [0]:
# ratings_clean(user_id, movie_id, rating, rating_ts, rating_date, rating_time)
# tags_clean(user_id, movie_id, tag, tags_ts, tags_date, tags_time)
# movies_clean(movie_id, title, release_year)
# movie_genres(movie_id, genre)

In [0]:
spark.sql("""select * from ratings_clean limit 2""").show()

In [0]:
# Q1. Create a unified ratings dataset with movie information.
# Join ratings_clean with movies_clean to get:
# movie title, release year, rating, rating_date.
# Hint:
# INNER JOIN on movie_i
spark.sql("""select title as movie_title,release_year,rating,rating_date from movies_clean m inner join ratings_clean r
          on m.movie_id == r.movie_id """).show()

In [0]:
# Q2. Which movies are the most popular by rating count?
# List top 10 movies with the highest number of ratings.
# Hint:
# GROUP BY movie_id, title
# ORDER BY COUNT(*) DESC
spark.sql("""select r.movie_id,title,count(*)as total_rating from ratings_clean r inner join movies_clean m on r.movie_id= m.movie_id 
          group by r.movie_id,title order by count(*) desc LIMIT 10 """).show()

In [0]:
# Q3. What is the average rating and total ratings per movie?
# Create a movie-level summary.
# Hint:
# Use AVG(rating) and COUNT(*)
spark.sql("""select r.movie_id,title,avg(rating)as Average_rating,sum(rating) as total_rating
           from ratings_clean r inner join movies_clean m 
          on r.movie_id = m.movie_id 
          group  by r.movie_id,title""").show()

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW movies_clean AS
SELECT
    movie_id,

    -- title without year
    regexp_replace(title, '\\\\s*\\\\(\\\\d{4}\\\\)', '') AS title,

    title AS original_title,
    genres,

    CAST(regexp_extract(title, '\\\\((\\\\d{4})\\\\)', 1) AS INT) AS release_year
FROM movies
""")


In [0]:
# Q4. Identify “Hidden Gem” movies.
# Movies with:
# - Average rating >= 4.5
# - Total ratings <= 50
# Hint:
# Use HAVING with both AVG and COUNT

spark.sql("""select r.movie_id,title,avg(rating)as Average_rating,count(rating)as Total_ratings from ratings_clean r inner join movies_clean m 
          on r.movie_id = m.movie_id 
          --where title=='Hidden Gem'
          group by r.movie_id,title
          having (avg(rating) >=4.5) and (count(rating) <=50) """).show(truncate=False)


In [0]:
# Q5. How do ratings vary by release year?
# Analyze average rating and rating volume by movie release year.
# Hint:
# GROUP BY release_year
spark.sql("""select release_year,avg(rating) as average_rating,count(rating)as rating_volume from movies_clean m inner join ratings_clean r 
            on m.movie_id=r.movie_id 
          group by release_year
          order by release_year """).show()

In [0]:
# Q6. What is the average rating per genre?
# Analyze user preference across genres.
# Hint:
# JOIN ratings_clean with movie_genres
# GROUP BY genre

spark.sql("""select m.genre,avg(r.rating) as avg_Rating from ratings_clean r inner join movie_genres m on r.movie_id= m.movie_id WHERE m.genre IS NOT NULL group by genre""").show()

In [0]:
# Q7. Which genres are the most popular?
# Rank genres based on total number of ratings received.
# Hint:
# COUNT(*) after joining ratings_clean and movie_genres

spark.sql("select genre,count('*') as total_number,RANK() OVER (ORDER BY COUNT(*) DESC) from ratings_clean r inner join movie_genres m on r.movie_id = m.movie_id group by genre order by total_number desc ").show()

In [0]:
# Q8. Which movies belong to the highest number of genres?
# Identify movies with multi-genre richness.
# Hint:
# COUNT(DISTINCT genre) GROUP BY movie_id

spark.sql("select movie_id,count(distinct genre) as number_genres from movie_genres group by movie_id").show()

In [0]:
# Q9. Compare popularity vs quality at genre level.
# For each genre calculate:
# - Average movie rating
# - Average ratings per movie
# Hint:
# Two-level aggregation:
# (movie → genre)
spark.sql("""select g.genre,AVG(avg_ratings.average_rating) AS avg_movie_rating,
                AVG(avg_ratings.rating_count) AS avg_ratings_per_movie  from (
          select movie_id,avg(rating) as average_rating,count(rating) as rating_count from ratings_clean
          group by movie_id
          order by movie_id asc) as avg_ratings join movie_genres g on avg_ratings.movie_id = g.movie_id GROUP BY g.genre
          """).show()


In [0]:
# Q10. Which genres produce more highly rated movies?
# Count movies with average rating >= 4.0 per genre.
# Hint:
# First compute movie-level avg rating,
# then aggregate by genre.
spark.sql("""select g.genre, count(distinct m.movie_id) as total_movie from(select movie_id,
          avg(rating) as avg from ratings_clean  group by movie_id having avg(rating)>=4.0 ) high_movie_rating join movies_clean m
          on high_movie_rating.movie_id=m.movie_id
          join movie_genres g on m.movie_id = g.movie_id 
            GROUP BY g.genre
            ORDER BY total_movie DESC
            """).show(truncate=False)


#  #**** PART 4 – Tags Analytics & User Intent Analysis

In [0]:
# ratings_clean(user_id, movie_id, rating, rating_ts, rating_date)
# tags_clean(user_id, movie_id, tag, tags_ts, tags_date)
# movies_clean(movie_id, title, release_year)
# movie_genres(movie_id, genre)

In [0]:
# Q1. What is the total number of tags after cleaning?
# Measure valid tagging activity volume.
spark.sql("select count(*) from tags_clean").show()

In [0]:
# Q2. Which tags are used most frequently?
# List the top 10 most common tags.
spark.sql("""select count(*) as total ,tag from tags_clean group by tag order by total desc Limit 10""").show(truncate=False)

In [0]:
# Q3. Which movies receive the highest number of tags?
# Identify the most tagged movies along with titles
spark.sql("""select t.movie_id, m.title, count(t.tag) as total_tag from tags_clean t join movies_clean m on t.movie_id = m.movie_id
           group by t.movie_id, m.title order by total_tag desc Limit 1 """).show()

In [0]:
# Q4. What is the average movie rating associated with each tag?
# Understand how user perception (tags) aligns with ratings.
spark.sql("""select avg(rating) as average_rating,tag
           from tags_clean t join ratings_clean r on t.movie_id = r.movie_id 
           group by tag """).show(truncate=False)

In [0]:
# Q5. Which tags are commonly associated with highly rated movies?
# Focus on movies with average rating >= 4.0.
spark.sql("""select count(*) as tag_frequantly,t.tag
           from (select movie_id ,avg(rating) as avg_rating from ratings_clean 
           group by movie_id
           having avg(rating) >=4.0)ratings_clean 
           join  tags_clean t  on t.movie_id = ratings_clean.movie_id 
           group by t.tag
           order by tag_frequantly desc """).show(truncate=False)

In [0]:
# Q6. Do users usually tag movies before or after rating them?
# Analyze the order of tagging vs rating events.
spark.sql("""select count(*) as total,action_order  from (select 
          case when t.tags_ts <r.rating_ts then "Before_rating"
            when t.tags_ts > r.rating_ts then "After_rating"
            ELSE "some thing Else"
            end as action_order
            from ratings_clean r join tags_clean t on r.movie_id=t.movie_id and r.user_id=t.user_id) group by action_order""").show()

In [0]:
# Q7. How much time typically passes between rating and tagging?
# Calculate time difference (in hours or days).
spark.sql("""select case when diff_hours <1 then "within 1 hours" 
                        when diff_hours < 24 then "within 1 days"
                        else "after 1 hours"
                        end AS bucket, count(*) as total from
                            (select rating_ts,rating_date,rating_time,tags_ts,tags_date,tags_time,
          (unix_timestamp(t.tags_ts) - unix_timestamp(r.rating_ts)) / 3600 AS diff_hours,
          (unix_timestamp(t.tags_ts) - unix_timestamp(r.rating_ts))/ 86400 As diff_days
             from ratings_clean r join tags_clean t on r.movie_id = t.movie_id and r.user_id = t.user_id) group by bucket""").show()

In [0]:
# Q8. Are there movies that are tagged but never rated?
# Identify movies that appear in tags_clean but not in ratings_clean.
spark.sql("""with cte1 as (
    select distinct movie_id from ratings_clean
),
cte2 as (
    select distinct movie_id from tags_clean
)
select c2.movie_id from cte2 c2 left anti join cte1 c1 on c1.movie_id = c2.movie_id 
          """).show()

In [0]:
# Q9. Who are the most active taggers?
# List top 10 users based on number of tags created.
spark.sql(""" select user_id,count(*) as number_of_tags from tags_clean group by user_id order by count(*) desc Limit 10  """).show()

In [0]:
# Q10. What is the tag density per movie?
# Calculate the ratio of total tags to total ratings for each movie.
# Hint:
# Aggregate tags_clean and ratings_clean separately by movie_id,
# then JOIN and compute (tag_count / rating_count)

spark.sql("""with cte as(select t.movie_id,count(tag)as total_tag from tags_clean t group by movie_id),
          cte2 as(select r.movie_id,count(rating)as total_rating from ratings_clean r group by movie_id)
          select cte.movie_id, total_Tag,total_rating,(total_tag/total_rating)as ratio from cte join cte2 on cte.movie_id= cte2.movie_id  order by ratio desc""").show()

#PART 5 – Window Functions & Time-Aware Analytics

In [0]:
# Q1. What is the running average rating for each movie over time?
# Track how movie perception evolves as new ratings arrive.
spark.sql("select  movie_id,rating,rating_ts, avg(rating) over(partition by movie_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)as Ratings_arrive from ratings_clean order by rating_ts,movie_id").show()

In [0]:
# # Q2. What was the first movie rated by each user?
# Identify user onboarding behavior.
spark.sql("select movie_id,user_id,row_number() over(partition by user_id order by rating_ts)  as user_behaviour from ratings_clean").show()


In [0]:
# What was the most recent movie rated by each user?
# Understand latest user activity.
spark.sql("""
          select user_id,movie_id,rating_ts from (select user_id,movie_id,rating_ts,
          row_number() over(partition by user_id order by rating_ts desc) as rn from ratings_clean)t
          where rn=1
          """).show()

In [0]:
spark.sql("""
SELECT user_id, movie_id, rating_ts
FROM (
    SELECT
        user_id,
        movie_id,
        rating_ts,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY rating_ts DESC
        ) AS rn
    FROM ratings_clean
) t
WHERE rn = 1
ORDER BY user_id
""").show()


In [0]:
# Q4. How does a user’s rating change compared to their previous rating?
# Measure rating consistency or drift.
spark.sql("""select movie_id,rating,lag(rating) over(partition by user_id order by rating_ts) as previous_rating,
                rating-lag(rating) over(partition by user_id order by rating_ts) as changed_rating 
                from ratings_clean""").show()

In [0]:
# Q5. Which movies have the most volatile ratings?
# Identify polarizing movies with high rating variance.
spark.sql("""select movie_id, variance(rating)v_rating , stddev(rating)s_Rating
           from ratings_clean group by movie_id order by variance(rating) desc""").show()

In [0]:
# Q6. Rank movies within each genre by average rating.
# Compare movies only within the same genre.
spark.sql("""
          with Movie_genre as(
              select movie_id,explode(split(genres,'\\|'))as genre from movies_clean
          ),
          Avg_rating as(
              select r.movie_id,avg(rating) as avg_rating,m.genre from ratings_clean r join Movie_genre m 
              on r.movie_id = m.movie_id
              group by r.movie_id,m.genre
          )
          select *,rank()over(partition by genre order by avg_rating desc)as rank_genre from Avg_rating
          """).show()

In [0]:
# Q7. What percentage of total ratings does each genre contribute?
# Measure genre share in overall engagement.percentage = (genre_ratings / total_ratings) * 100

spark.sql("""
          with genre_contribute as(
          select genre,count(*)genre_ratings from movie_genres m join ratings_clean r on m.movie_id=r.movie_id 
          group by genre),
          percentage_genre as(
              select genre,genre_ratings,(genre_ratings * 100 / sum(genre_ratings)over())as percentage_rating from genre_contribute
              group by genre,genre_ratings
          )
          select * from percentage_genre order by percentage_rating desc;
          """).show() 

In [0]:
# Q8. When did each movie receive its first rating?
# Understand movie lifecycle starting point.
spark.sql("select movie_id,min(rating_ts)as first_rating from ratings_clean group by movie_id order by movie_id").show()


In [0]:
# Q9. How long did it take for each movie to receive its first 10 ratings?
# Measure early traction speed.
spark.sql("""
          with row_num AS(select  movie_id,rating_ts, row_number() over(partition by movie_id order by rating_ts) as rn from ratings_clean) ,
          first_10 as(
              select movie_id,min(case when rn=1 then rating_ts end) as first_rating_ts,
                    min(case when rn=10 then rating_ts end) as 10th_rating_ts
                    from row_num where rn<=10
                    group by movie_id
          )
          select movie_id,first_rating_ts,10th_rating_ts ,(10th_rating_ts-first_rating_ts) as diff_rating 
          from  first_10 order by movie_id
           """).show(truncate=False)

In [0]:
# Q10. Identify users with very consistent rating behavior.
# Find users whose rating variance is very low.
# Hint:
# GROUP BY user_id
# Use VARIANCE(rating)

spark.sql("""select user_id,variance(rating)as vari_rating from ratings
           group by user_id order by vari_rating asc """).show()

# PART 6 – Business Scenarios & Decision-Driven Analytics

In [0]:
# Q1. Identify “Binge Raters”.
# Users who rated 20 or more movies on the same day.

spark.sql("""select user_id,count(rating) as rated,rating_date from ratings_clean 
          group by rating_date,user_id
          having count(rating) >=20
          order by user_id
          """).show()

In [0]:
# Q2. Identify “Consistent Raters”.
# Users whose ratings show very low variance.
spark.sql("""select user_id,variance(rating)as var_rating
           from ratings_clean group by user_id
           order by var_rating""").show()

In [0]:
# Q3. Identify “Extreme Raters”.
# Users who mostly give very low (<= 2.0) or very high (>= 4.5) ratings.
spark.sql("""
          select user_id,
            count(*) as total_rating,
            case when rating <= 2.0 then "very_low"
                when rating >=4.5 then "High_low"
                else "Mediam"
            end as Rating_extreme_raters,
            SUM(CASE WHEN rating <= 2.0 OR rating >= 4.5 THEN 1 ELSE 0 END) AS extreme_ratings
           from ratings_clean
           group by user_id,rating
          """).show()

             

In [0]:
# Q4. Which movies gained popularity very quickly?
# Movies that reached their first 100 ratings in the shortest time.
#movie  (100 rating  sbse km time m hui)

spark.sql("""
          with ratings_popular as (
              select movie_id,rating_ts,row_number()over(partition by movie_id order by rating_ts) as rn from ratings_clean
          ),
          100_ratings as(
              select movie_id,MIN(case when rn =1 then rating_ts END )as first_rating,
              MIN(case when rn=100 then rating_ts END) as 100_rating
              from ratings_popular group by movie_id
          )
          select movie_id,first_rating,100_rating,(100_rating-first_rating)as diff_time from 100_ratings
          """).show(truncate=False)

In [0]:
# Q5. Identify “Hidden Gems”.
# Movies with:
# - Average rating >= 4.5
# - Total ratings <= 50

spark.sql("""
          select movie_id,avg(rating)as Average_rating , count(rating)as total_rating
           from ratings_clean group by movie_id having avg(rating) >= 4.5 and count(rating) <=50 order by movie_id
          """).show()

In [0]:
# Q6. Identify “Polarizing Movies”.
# Movies with high disagreement among users.
spark.sql("""
          select movie_id,variance(rating)as high_variance from ratings_clean
          group by movie_id order by high_variance DESC
""").show(truncate=False)
       

In [0]:
# Q7. Which genres are mostly rated at night?
# Analyze user behavior during late hours.
spark.sql("""select genre,count(*) as night_late
           from ratings_clean r join movie_genres m on r.movie_id= m.movie_id
           where hour(rating_ts) >21 or hour(rating_ts) < 5
           group by genre
           order by night_late desc
            """).show()



In [0]:
# Q8. Which movies show rating improvement over time?
# Movies whose recent average rating is higher than early average rating.
spark.sql("""
WITH rating_order AS (
    SELECT movie_id,rating,rating_ts,ROW_NUMBER() OVER ( PARTITION BY movie_id ORDER BY rating_ts) AS rn,
        COUNT(*) OVER (PARTITION BY movie_id) AS total_cnt FROM ratings_clean),
early_recent AS (SELECT movie_id,CASE WHEN rn <= total_cnt * 0.5 THEN 'early'ELSE 'recent'END AS period,rating FROM rating_order
)
SELECT movie_id, AVG(CASE WHEN period = 'early' THEN rating END) AS early_avg_rating,AVG(CASE WHEN period = 'recent' THEN rating END) AS recent_avg_rating FROM early_recent GROUP BY movie_id HAVING recent_avg_rating > early_avg_rating ORDER BY (recent_avg_rating - early_avg_rating) DESC
""").show(truncate=False)


In [0]:
# Q9. Identify “Genre-Loyal” users.
# Users who give 80% or more of their ratings within a single genre.
# Hint:
# Join ratings_clean with movie_genres
# Use conditional aggregation
spark.sql("""
WITH user_genre_counts AS (
    SELECT r.user_id,g.genre,COUNT(*) AS genre_ratings FROM ratings_clean r JOIN movie_genres g ON r.movie_id = g.movie_id
    GROUP BY r.user_id, g.genre
),
user_total_counts AS (
    SELECT user_id,SUM(genre_ratings) AS total_ratings FROM user_genre_counts GROUP BY user_id
)
SELECT ug.user_id,ug.genre,ug.genre_ratings,ut.total_ratings,ug.genre_ratings / ut.total_ratings AS genre_ratio
FROM user_genre_counts ug JOIN user_total_counts ut ON ug.user_id = ut.user_id WHERE ug.genre_ratings / ut.total_ratings >= 0.80
""").show(truncate=False)


In [0]:
# Q10. Build a final analytical movie summary view.
# Create a reusable view containing:
# - movie_id
# - title
# - release_year
# - total_ratings
# - avg_rating
# - first_rating_date
# - last_rating_date
# - top_tag
# Hint:
# Use multiple CTEs, window functions for top_tag,
# and joins across ratings_clean, tags_clean, and movies_clean
spark.sql("""
            -- with final_analytical as(
                select m.movie_id,title,release_year,sum(rating)as total_ratings,avg(rating) as avg_rating,min(rating_ts) as first_rating_ts,max(rating_ts) as last_rating_ts,tag 
                from movies_clean as m join ratings_clean r on m.movie_id=r.movie_id
                join tags_clean as t on m.movie_id=t.movie_id
                group by m.movie_id,title,release_year,tag
          """).show()

In [0]:
spark.sql("""
          with rating_summary as(
              select movie_id,COUNT(*) AS total_ratings,AVG(rating) AS avg_rating,MIN(rating_ts) AS first_rating_date,MAX(rating_ts) AS last_rating_date FROM ratings_clean GROUP BY movie_id
          ),
          tag_rank as(
              select movie_id,tag,count(*) tag_Count,row_number()over(partition by movie_id order by count(*) desc)as rn from tags_clean
              group by movie_id,tag
          ),
          final as(
              select m.movie_id,m.title,m.release_year, r.total_ratings,r.avg_rating,r.first_rating_date,r.last_rating_date,t.tag AS top_tag
    FROM movies_clean m LEFT JOIN rating_summary r ON m.movie_id = r.movie_id LEFT JOIN tag_rank t
        ON m.movie_id = t.movie_id AND t.rn = 1
)
SELECT * FROM final;
          
          """).show()

In [0]:
spark.sql("show tables").show()